In [1]:
!pip install fairlearn
"""
PS26 – Auditing Fairness in AI-Based Resource Allocation
Idea #3 | Full Analysis Notebook
RQ1–RQ5 | German Credit + Adult Census Datasets
"""

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.pipeline import Pipeline

from fairlearn.metrics import (
    demographic_parity_difference,
    equalized_odds_difference,
    MetricFrame
)
from fairlearn.reductions import ExponentiatedGradient, DemographicParity
from fairlearn.postprocessing import ThresholdOptimizer

# ─── Output directory ────────────────────────────────────────────────────────
os.makedirs('/kaggle/working/figures', exist_ok=True)
os.makedirs('/kaggle/working/tables', exist_ok=True)

# ─── Color palette ───────────────────────────────────────────────────────────
COLORS = {
    'lr':  '#2196F3',   # blue  – Logistic Regression
    'rf':  '#4CAF50',   # green – Random Forest
    'xgb': '#FF9800',   # orange– XGBoost (GradientBoosting)
    'good':'#66BB6A',
    'bad': '#EF5350',
    'before': '#EF5350',
    'after':  '#66BB6A',
    'accent': '#7C4DFF',
}

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
})

# ══════════════════════════════════════════════════════════════════════════════
# DATA LOADING & PREPROCESSING
# ══════════════════════════════════════════════════════════════════════════════

print("Loading datasets...")

# ── German Credit ──
gc = pd.read_csv('/kaggle/input/datasets/akash06shaw/fairness-audit-datasets/german_credit_data.csv', index_col=0)

# Fill missing with mode
gc['Saving accounts'] = gc['Saving accounts'].fillna(gc['Saving accounts'].mode()[0])
gc['Checking account'] = gc['Checking account'].fillna(gc['Checking account'].mode()[0])

# Encode target
gc['target'] = (gc['Risk'] == 'bad').astype(int)   # 1 = bad (unfavourable)

# Protected attribute
gc['sex_bin'] = (gc['Sex'] == 'female').astype(int)  # 1 = female
gc['age_group'] = pd.cut(gc['Age'], bins=[0,30,45,100],
                          labels=['Young(<30)','Middle(30-45)','Senior(45+)'])

# Feature encoding
le = LabelEncoder()
cat_cols = ['Sex','Housing','Saving accounts','Checking account','Purpose']
for c in cat_cols:
    gc[c+'_enc'] = le.fit_transform(gc[c].astype(str))

gc_features = ['Age','Job','Credit amount','Duration',
               'Sex_enc','Housing_enc','Saving accounts_enc',
               'Checking account_enc','Purpose_enc']

X_gc = gc[gc_features]
y_gc = gc['target']
sex_gc = gc['sex_bin']

X_tr, X_te, y_tr, y_te, s_tr, s_te = train_test_split(
    X_gc, y_gc, sex_gc, test_size=0.25, random_state=42, stratify=y_gc)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)

# ── Adult Census ──
ad = pd.read_csv('/kaggle/input/datasets/akash06shaw/fairness-audit-datasets/adult.csv')
ad.columns = ad.columns.str.strip()
for col in ad.select_dtypes('object').columns:
    ad[col] = ad[col].str.strip()

ad = ad[ad['race'].isin(['White','Black'])]   # focus on 2 largest groups
ad['target'] = (ad['income'] == '>50K').astype(int)
ad['sex_bin'] = (ad['sex'] == 'Female').astype(int)
ad['race_bin'] = (ad['race'] == 'Black').astype(int)
ad['age_group'] = pd.cut(ad['age'], bins=[0,30,45,100],
                          labels=['Young(<30)','Middle(30-45)','Senior(45+)'])

ad_cat = ['workclass','education','marital.status','occupation',
          'relationship','race','sex','native.country']
for c in ad_cat:
    ad[c+'_enc'] = le.fit_transform(ad[c].astype(str))

ad = ad.reset_index(drop=True)   # ensure clean 0-based index before split

ad_features = ['age','education.num','hours.per.week','capital.gain',
               'capital.loss','workclass_enc','marital.status_enc',
               'occupation_enc','relationship_enc','race_enc','sex_enc']

X_ad = ad[ad_features]
y_ad = ad['target']

X_tr_ad, X_te_ad, y_tr_ad, y_te_ad = train_test_split(
    X_ad, y_ad, test_size=0.25, random_state=42, stratify=y_ad)

scaler_ad = StandardScaler()
X_tr_ad_sc = scaler_ad.fit_transform(X_tr_ad)
X_te_ad_sc = scaler_ad.transform(X_te_ad)

print("Datasets loaded and preprocessed.\n")

# ══════════════════════════════════════════════════════════════════════════════
# RQ1 – Model Fairness Comparison
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("RQ1: How do LR, RF, XGBoost differ in fairness metrics?")
print("=" * 60)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':             GradientBoostingClassifier(n_estimators=100, random_state=42),
}

rq1_results = {}
for name, model in models.items():
    model.fit(X_tr_sc, y_tr)
    y_pred = model.predict(X_te_sc)
    dpd = demographic_parity_difference(y_te, y_pred, sensitive_features=s_te)
    eod = equalized_odds_difference(y_te, y_pred, sensitive_features=s_te)
    acc = accuracy_score(y_te, y_pred)
    f1  = f1_score(y_te, y_pred)
    rq1_results[name] = {'Accuracy': acc, 'F1 Score': f1,
                         'Dem. Parity Diff': abs(dpd),
                         'Eq. Odds Diff': abs(eod)}
    print(f"  {name}: Acc={acc:.3f} | F1={f1:.3f} | DPD={abs(dpd):.3f} | EOD={abs(eod):.3f}")

# ── Figure RQ1 ──
rq1_df = pd.DataFrame(rq1_results).T
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('RQ1: Fairness Metrics Across ML Models\n(German Credit Dataset, n=1000)',
             fontsize=13, fontweight='bold', y=1.02)

metrics_perf  = ['Accuracy', 'F1 Score']
metrics_fair  = ['Dem. Parity Diff', 'Eq. Odds Diff']
model_colors  = [COLORS['lr'], COLORS['rf'], COLORS['xgb']]
model_names   = list(rq1_df.index)
x = np.arange(2)
width = 0.25

for ax, metric_group, title, ylabel in [
    (axes[0], metrics_perf,  'Predictive Performance',   'Score'),
    (axes[1], metrics_fair,  'Fairness Metrics\n(lower = fairer)', 'Difference (|value|)'),
]:
    for i, (mname, color) in enumerate(zip(model_names, model_colors)):
        vals = [rq1_df.loc[mname, m] for m in metric_group]
        bars = ax.bar(x + i*width, vals, width, label=mname,
                      color=color, edgecolor='white', linewidth=0.8)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x + width)
    ax.set_xticklabels(metric_group, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylim(0, max(ax.get_ylim()[1], 0.6))

plt.tight_layout()
plt.savefig('/kaggle/working/figures/RQ1_fairness_comparison.pdf', bbox_inches='tight')
plt.savefig('/kaggle/working/figures/RQ1_fairness_comparison.png', bbox_inches='tight', dpi=150)
plt.close()

# ── Table RQ1 ──
rq1_table = rq1_df.round(4).reset_index().rename(columns={'index': 'Model'})
rq1_table.to_csv('/kaggle/working/tables/RQ1_model_comparison.csv', index=False)
print("  → Figure & table saved.\n")


# ══════════════════════════════════════════════════════════════════════════════
# RQ2 – SHAP Feature Analysis
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("RQ2: Which features act as proxies for protected attributes?")
print("=" * 60)

# Use Random Forest for SHAP (best balance)
rf_model = models['Random Forest']
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_te)

# Handle both old (list) and new (3D array) shap output formats
if isinstance(shap_values, list):
    sv = shap_values[1]                    # old: list of [class0, class1]
elif shap_values.ndim == 3:
    sv = shap_values[:, :, 1]             # new: (n_samples, n_features, n_classes)
else:
    sv = shap_values

feature_names_clean = ['Age', 'Job', 'Credit Amt', 'Duration',
                        'Sex', 'Housing', 'Saving Acct',
                        'Checking Acct', 'Purpose']

mean_shap = np.abs(sv).mean(axis=0)
shap_df = pd.DataFrame({'Feature': feature_names_clean,
                         'Mean |SHAP|': mean_shap}).sort_values('Mean |SHAP|', ascending=False)

# Protected attribute flag
protected = {'Sex', 'Age'}
shap_df['Protected Attribute?'] = shap_df['Feature'].apply(
    lambda x: '✓ Yes' if x in protected else 'No')
shap_df['Rank'] = range(1, len(shap_df)+1)

print(shap_df[['Rank','Feature','Mean |SHAP|','Protected Attribute?']].to_string(index=False))

# ── Figure RQ2 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('RQ2: SHAP Feature Importance & Proxy Detection\n(Random Forest on German Credit Dataset)',
             fontsize=13, fontweight='bold', y=1.02)

# Left: bar chart
colors_shap = [COLORS['bad'] if f in protected else COLORS['lr']
               for f in shap_df['Feature']]
bars = axes[0].barh(shap_df['Feature'][::-1], shap_df['Mean |SHAP|'][::-1],
                    color=colors_shap[::-1], edgecolor='white')
for bar, val in zip(bars, shap_df['Mean |SHAP|'][::-1]):
    axes[0].text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9)
axes[0].set_xlabel('Mean |SHAP Value|', fontsize=10)
axes[0].set_title('Feature Importance by SHAP', fontsize=11, fontweight='bold')
patch1 = mpatches.Patch(color=COLORS['bad'],  label='Protected Attribute')
patch2 = mpatches.Patch(color=COLORS['lr'],   label='Non-protected Feature')
axes[0].legend(handles=[patch1, patch2], fontsize=9)

# Right: SHAP beeswarm-style summary using dot plot
shap.summary_plot(sv, X_te, feature_names=feature_names_clean,
                  show=False, plot_size=None, color_bar=True)
fig2 = plt.gcf()
fig2.set_size_inches(7, 6)
fig2.suptitle('SHAP Summary Plot\n(impact direction per sample)',
              fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/figures/RQ2_shap_summary.pdf', bbox_inches='tight')
plt.savefig('/kaggle/working/figures/RQ2_shap_summary.png', bbox_inches='tight', dpi=150)
plt.close('all')

# Bar chart standalone (cleaner for poster)
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(shap_df['Feature'][::-1], shap_df['Mean |SHAP|'][::-1],
        color=colors_shap[::-1], edgecolor='white', linewidth=0.8)
for i, (val, feat) in enumerate(zip(shap_df['Mean |SHAP|'][::-1],
                                     shap_df['Feature'][::-1])):
    ax.text(val + 0.001, i, f'{val:.4f}', va='center', fontsize=9)
ax.set_xlabel('Mean |SHAP Value| (feature impact on prediction)', fontsize=10)
ax.set_title('RQ2: SHAP Feature Importance — Protected Attribute Proxy Detection\n(Random Forest, German Credit Dataset)',
             fontsize=11, fontweight='bold')
ax.legend(handles=[patch1, patch2], fontsize=9)
plt.tight_layout()
plt.savefig('/kaggle/working/figures/RQ2_shap_bar.pdf', bbox_inches='tight')
plt.savefig('/kaggle/working/figures/RQ2_shap_bar.png', bbox_inches='tight', dpi=150)
plt.close()

# ── Table RQ2 ──
shap_df[['Rank','Feature','Mean |SHAP|','Protected Attribute?']].to_csv(
    '/kaggle/working/tables/RQ2_shap_features.csv', index=False)
print("  → Figure & table saved.\n")


# ══════════════════════════════════════════════════════════════════════════════
# RQ3 – Reweighing Mitigation
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("RQ3: How does reweighing affect fairness metrics?")
print("=" * 60)

# Baseline (Random Forest, no mitigation)
rf_base = RandomForestClassifier(n_estimators=100, random_state=42)
rf_base.fit(X_tr_sc, y_tr)
y_pred_base = rf_base.predict(X_te_sc)

dpd_base = abs(demographic_parity_difference(y_te, y_pred_base, sensitive_features=s_te))
eod_base = abs(equalized_odds_difference(y_te, y_pred_base, sensitive_features=s_te))
f1_base  = f1_score(y_te, y_pred_base)
acc_base = accuracy_score(y_te, y_pred_base)

# Mitigated — ExponentiatedGradient with DemographicParity constraint
base_estimator = RandomForestClassifier(n_estimators=100, random_state=42)
mitigator = ExponentiatedGradient(base_estimator, constraints=DemographicParity())
mitigator.fit(X_tr_sc, y_tr, sensitive_features=s_tr)
y_pred_mit = mitigator.predict(X_te_sc)

dpd_mit = abs(demographic_parity_difference(y_te, y_pred_mit, sensitive_features=s_te))
eod_mit = abs(equalized_odds_difference(y_te, y_pred_mit, sensitive_features=s_te))
f1_mit  = f1_score(y_te, y_pred_mit)
acc_mit = accuracy_score(y_te, y_pred_mit)

print(f"  Baseline  — F1:{f1_base:.3f} | DPD:{dpd_base:.3f} | EOD:{eod_base:.3f}")
print(f"  Mitigated — F1:{f1_mit:.3f}  | DPD:{dpd_mit:.3f}  | EOD:{eod_mit:.3f}")

# ── Figure RQ3 ──
metrics_rq3  = ['Dem. Parity Diff', 'Eq. Odds Diff', 'F1 Score']
base_vals    = [dpd_base, eod_base, f1_base]
mit_vals     = [dpd_mit,  eod_mit,  f1_mit]

x = np.arange(len(metrics_rq3))
width = 0.3

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, base_vals, width, label='Baseline (No Mitigation)',
               color=COLORS['before'], edgecolor='white', linewidth=0.8)
bars2 = ax.bar(x + width/2, mit_vals,  width, label='After Reweighing (DemParity Constraint)',
               color=COLORS['after'],  edgecolor='white', linewidth=0.8)

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metrics_rq3, fontsize=11)
ax.set_ylabel('Metric Value', fontsize=11)
ax.set_title('RQ3: Effect of Fairness Mitigation on Demographic Parity & Model Performance\n'
             '(Reweighing via ExponentiatedGradient — German Credit Dataset)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, max(max(base_vals), max(mit_vals)) * 1.2)

# Annotation arrows for fairness metrics
for i in range(2):
    reduction = ((base_vals[i] - mit_vals[i]) / base_vals[i]) * 100 if base_vals[i] > 0 else 0
    ax.annotate(f'{reduction:.1f}% ↓', xy=(i + width/2, mit_vals[i] + 0.02),
                fontsize=9, color='darkgreen', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/figures/RQ3_mitigation_comparison.pdf', bbox_inches='tight')
plt.savefig('/kaggle/working/figures/RQ3_mitigation_comparison.png', bbox_inches='tight', dpi=150)
plt.close()

# ── Table RQ3 ──
rq3_table = pd.DataFrame({
    'Condition': ['Baseline', 'After Reweighing', '% Change'],
    'Accuracy':          [f'{acc_base:.4f}', f'{acc_mit:.4f}',
                          f'{((acc_mit-acc_base)/acc_base)*100:.2f}%'],
    'F1 Score':          [f'{f1_base:.4f}',  f'{f1_mit:.4f}',
                          f'{((f1_mit-f1_base)/f1_base)*100:.2f}%'],
    'Dem. Parity Diff':  [f'{dpd_base:.4f}', f'{dpd_mit:.4f}',
                          f'{((dpd_mit-dpd_base)/dpd_base)*100:.2f}%'],
    'Eq. Odds Diff':     [f'{eod_base:.4f}', f'{eod_mit:.4f}',
                          f'{((eod_mit-eod_base)/eod_base)*100:.2f}%'],
})
rq3_table.to_csv('/kaggle/working/tables/RQ3_mitigation_results.csv', index=False)
print("  → Figure & table saved.\n")


# ══════════════════════════════════════════════════════════════════════════════
# RQ4 – Fairness–Accuracy Trade-off
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("RQ4: What is the trade-off between F1 and fairness?")
print("=" * 60)

# Vary decision threshold for each model and compute F1 vs DPD
def threshold_sweep(model, X_test, y_test, sensitive, thresholds):
    results = []
    if hasattr(model, 'predict_proba'):
        probs = model.predict_proba(X_test)[:, 1]
    else:
        probs = model.decision_function(X_test)
        probs = (probs - probs.min()) / (probs.max() - probs.min())
    for t in thresholds:
        y_pred = (probs >= t).astype(int)
        if len(np.unique(y_pred)) < 2:
            continue
        f1  = f1_score(y_test, y_pred, zero_division=0)
        dpd = abs(demographic_parity_difference(y_test, y_pred, sensitive_features=sensitive))
        results.append({'threshold': t, 'F1': f1, 'DPD': dpd})
    return pd.DataFrame(results)

thresholds = np.linspace(0.2, 0.8, 25)
sweep_models = {
    'Logistic Regression': models['Logistic Regression'],
    'Random Forest':       models['Random Forest'],
    'XGBoost':             models['XGBoost'],
}
sweep_colors = [COLORS['lr'], COLORS['rf'], COLORS['xgb']]

fig, ax = plt.subplots(figsize=(10, 7))
rq4_all = []

for (mname, model), color in zip(sweep_models.items(), sweep_colors):
    sweep = threshold_sweep(model, X_te_sc, y_te, s_te, thresholds)
    ax.scatter(sweep['DPD'], sweep['F1'], color=color, alpha=0.6, s=50, label=mname)
    # Mark the default threshold point
    default = sweep.iloc[(sweep['threshold'] - 0.5).abs().argsort()[:1]]
    ax.scatter(default['DPD'], default['F1'], color=color, s=150,
               edgecolors='black', linewidths=1.5, zorder=5)
    sweep['Model'] = mname
    rq4_all.append(sweep)

rq4_df = pd.concat(rq4_all)
ax.set_xlabel('Demographic Parity Difference (|DPD|) — lower = fairer', fontsize=11)
ax.set_ylabel('F1 Score — higher = better', fontsize=11)
ax.set_title('RQ4: Fairness–Performance Trade-off Across ML Models\n'
             '(Each point = one decision threshold; ● = default threshold 0.5)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=10)

# Quadrant lines
ax.axhline(rq4_df['F1'].median(), color='grey', linestyle='--', alpha=0.5, linewidth=0.8)
ax.axvline(rq4_df['DPD'].median(), color='grey', linestyle='--', alpha=0.5, linewidth=0.8)
ax.text(0.02, 0.95, 'High F1\nLow Bias\n(Ideal)', transform=ax.transAxes,
        fontsize=8, color='green', va='top', alpha=0.7)
ax.text(0.75, 0.05, 'Low F1\nHigh Bias\n(Worst)', transform=ax.transAxes,
        fontsize=8, color='red', va='bottom', alpha=0.7)

plt.tight_layout()
plt.savefig('/kaggle/working/figures/RQ4_tradeoff_curve.pdf', bbox_inches='tight')
plt.savefig('/kaggle/working/figures/RQ4_tradeoff_curve.png', bbox_inches='tight', dpi=150)
plt.close()

# ── Table RQ4 (default threshold summary) ──
rq4_summary = rq4_df[rq4_df['threshold'].between(0.49, 0.51)].groupby('Model').first().reset_index()
rq4_summary = rq4_summary[['Model','F1','DPD']].round(4)
rq4_summary.columns = ['Model', 'F1 Score', 'Dem. Parity Diff']
rq4_summary['Fairness-Accuracy Ratio'] = (rq4_summary['F1 Score'] /
                                           (rq4_summary['Dem. Parity Diff'] + 1e-6)).round(2)
rq4_summary.to_csv('/kaggle/working/tables/RQ4_tradeoff_summary.csv', index=False)
print(rq4_summary.to_string(index=False))
print("  → Figure & table saved.\n")


# ══════════════════════════════════════════════════════════════════════════════
# RQ5 – Intersectional Fairness (Adult Census)
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("RQ5: Do aggregate metrics mask intersectional disparities?")
print("=" * 60)

# Train RF on Adult Census
rf_ad = RandomForestClassifier(n_estimators=100, random_state=42)
rf_ad.fit(X_tr_ad_sc, y_tr_ad)
y_pred_ad = rf_ad.predict(X_te_ad_sc)

# Build intersectional subgroup labels
ad_test = ad.iloc[X_te_ad.index].copy()
ad_test['y_pred'] = y_pred_ad
ad_test['y_true'] = y_te_ad.values
ad_test['subgroup'] = ad_test['sex'].str.strip() + '\n' + ad_test['age_group'].astype(str)

subgroup_stats = []
for sg, grp in ad_test.groupby('subgroup'):
    if len(grp) < 20:
        continue
    pos_rate = grp['y_pred'].mean()
    true_pos  = grp['y_true'].mean()
    f1_sg    = f1_score(grp['y_true'], grp['y_pred'], zero_division=0)
    subgroup_stats.append({
        'Subgroup': sg,
        'n': len(grp),
        'Predicted Positive Rate': pos_rate,
        'Actual Positive Rate': true_pos,
        'F1 Score': f1_sg,
    })

sg_df = pd.DataFrame(subgroup_stats).sort_values('Predicted Positive Rate', ascending=False)
print(sg_df.to_string(index=False))

# Aggregate (overall)
overall_ppr = y_pred_ad.mean()
print(f"\n  Overall predicted positive rate: {overall_ppr:.3f}")
print(f"  Max subgroup deviation: {abs(sg_df['Predicted Positive Rate'] - overall_ppr).max():.3f}")

# ── Figure RQ5 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('RQ5: Intersectional Fairness — Do Aggregate Metrics Hide Subgroup Disparities?\n'
             '(Random Forest on Adult Census Dataset — White & Black subpopulations)',
             fontsize=11, fontweight='bold', y=1.02)

# Left: predicted positive rate by subgroup
sg_sorted = sg_df.sort_values('Predicted Positive Rate')
bar_colors = [COLORS['lr'] if 'Female' in s else COLORS['rf'] for s in sg_sorted['Subgroup']]
bars = axes[0].barh(sg_sorted['Subgroup'], sg_sorted['Predicted Positive Rate'],
                    color=bar_colors, edgecolor='white', linewidth=0.8)
axes[0].axvline(overall_ppr, color='red', linestyle='--', linewidth=1.5,
                label=f'Overall avg: {overall_ppr:.3f}')
for bar, val in zip(bars, sg_sorted['Predicted Positive Rate']):
    axes[0].text(val + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)
axes[0].set_xlabel('Predicted Positive Rate', fontsize=10)
axes[0].set_title('Predicted Positive Rate by Intersectional Subgroup', fontsize=10, fontweight='bold')
axes[0].legend(fontsize=9)
patch_f = mpatches.Patch(color=COLORS['lr'], label='Female')
patch_m = mpatches.Patch(color=COLORS['rf'], label='Male')
axes[0].legend(handles=[patch_f, patch_m,
               mpatches.Patch(color='red', label=f'Overall avg: {overall_ppr:.3f}')],
               fontsize=9)

# Right: F1 Score by subgroup
sg_sorted2 = sg_df.sort_values('F1 Score')
bar_colors2 = [COLORS['lr'] if 'Female' in s else COLORS['rf'] for s in sg_sorted2['Subgroup']]
bars2 = axes[1].barh(sg_sorted2['Subgroup'], sg_sorted2['F1 Score'],
                     color=bar_colors2, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars2, sg_sorted2['F1 Score']):
    axes[1].text(val + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)
axes[1].set_xlabel('F1 Score', fontsize=10)
axes[1].set_title('F1 Score by Intersectional Subgroup', fontsize=10, fontweight='bold')
axes[1].legend(handles=[patch_f, patch_m], fontsize=9)

plt.tight_layout()
plt.savefig('/kaggle/working/figures/RQ5_intersectional_fairness.pdf', bbox_inches='tight')
plt.savefig('/kaggle/working/figures/RQ5_intersectional_fairness.png', bbox_inches='tight', dpi=150)
plt.close()

# ── Table RQ5 ──
sg_df_out = sg_df.copy()
sg_df_out['Subgroup'] = sg_df_out['Subgroup'].str.replace('\n', ' + ')
sg_df_out = sg_df_out.round(4)
sg_df_out.to_csv('/kaggle/working/tables/RQ5_intersectional_subgroups.csv', index=False)
print("  → Figure & table saved.\n")


# ══════════════════════════════════════════════════════════════════════════════
# DONE
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("ALL DONE — Summary of outputs:")
print("=" * 60)
print("\nFIGURES:")
for f in sorted(os.listdir('/kaggle/working/figures')):
    print(f"  /kaggle/working/figures/{f}")
print("\nTABLES:")
for f in sorted(os.listdir('/kaggle/working/tables')):
    print(f"  /kaggle/working/tables/{f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 3.3 MB/s eta 0:00:0000:01
Loading datasets...
Datasets loaded and preprocessed.

RQ1: How do LR, RF, XGBoost differ in fairness metrics?
  Logistic Regression: Acc=0.712 | F1=0.217 | DPD=0.069 | EOD=0.081
  Random Forest: Acc=0.752 | F1=0.456 | DPD=0.034 | EOD=0.037
  XGBoost: Acc=0.756 | F1=0.440 | DPD=0.082 | EOD=0.194
  → Figure & table saved.

RQ2: Which features act as proxies for protected attributes?
 Rank       Feature  Mean |SHAP| Protected Attribute?
    1    Credit Amt     0.173405                   No
    2      Duration     0.080516                   No
    3           Age     0.053419                ✓ Yes
    4 Checking Acct     0.045110                   No
    5       Housing     0.042074                   No
    6       Purpose     0.024244                   No
    7   Saving Acct     0.008836                   No
    8           Job     0.008271                   No
    9           Sex     0.004309            